[![StackExchange Codes](https://cdn.sstatic.net/Sites/datascience/Img/logo.svg?v=4a36350d1199)](https://github.com/RoyiAvital/StackExchangeCodes)

# StackExchange Data Science  

## Q107366 - Find a Discrete Distribution which Maximize Entropy with Constraints on Mean Value
[Find a Discrete Distribution which Maximize Entropy with Constraints on Mean Value](https://datascience.stackexchange.com/questions/107366)

> Notebook by:
> - Royi Avital RoyiAvital@yahoo.com

## Revision History

| Version | Date       | User        |Content / Changes                                                   |
|---------|------------|-------------|--------------------------------------------------------------------|
| 1.0.000 | 20/06/2026 | Royi Avital | First version                                                      |

In [ ]:
# Import Packages

# General Tools
import numpy as np
import scipy as sp
import pandas as pd

# Deep Learning
import torch

# Optimization
import cvxpy as cp

# Miscellaneous
import math
from platform import python_version
import random

# Typing
from typing import Callable, List, Optional, Tuple, Union
from numpy.typing import NDArray
from torch import Tensor

# Visualization
import matplotlib.pyplot as plt

# Jupyter
from IPython import get_ipython

In [ ]:
# Configuration

# warnings.filterwarnings("ignore")

seedNum = 512
np.random.seed(seedNum)
random.seed(seedNum)

# Matplotlib default color palette
lMatPltLibclr = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']

In [ ]:
# Auxiliary Functions

def MaxEntropyDistributionCVXPY( vVals: NDArray, meanValue: float ) -> NDArray:
    """
    Finds the discrete distribution which maximizes Entropy with the given mean value using CVXPY.
    
    Parameters:
    - vVals: Array of values for the discrete distribution.
    - meanValue: Desired mean value of the distribution.
    Returns:
    - vP: Optimization variables.
    """

    numElements = len(vVals)
    
    # Define the variable
    vP        = cp.Variable(numElements)
    cpObjFun  = cp.Maximize(cp.sum(cp.entr(vP)))
    # Define Constraints
    cpConst   = [cp.sum(vP) == 1, vP >= 0, (vP @ vVals) == meanValue]
    # Define the Problem
    oCvxPrb   = cp.Problem(cpObjFun, cpConst)
    
    # Solve the problem
    # oCvxPrb.solve(solver = cp.SCS, verbose = False)
    oCvxPrb.solve(solver = cp.CLARABEL, verbose = False)
    
    return vP.value

In [ ]:
# Parameters

vVals       = np.arange(1, 7) #<! Dice
numElements = len(vVals)
meanValue   = 4.5

# Solution
numIter = 10_000
μ       = 5e-5

In [ ]:
# Load Data


In [ ]:
# Plot Data


## Reference Solution

In [ ]:
# Solution by CVXPY

vPRef = MaxEntropyDistributionCVXPY(vVals, meanValue)
print(f'Optimal Distribution: {vPRef}')
print(f'Non Negativity: {np.all(vPRef >= 0)}')
print(f'Unit Sum: {np.sum(vPRef)}')
print(f'Mean Value: {vPRef @ vVals}')
print(f'Entropy: {sp.stats.entropy(vPRef)}')

## PyTorch Solution

In [ ]:
# Elements

mA = torch.tensor([(1.0, ) * numElements, vVals], dtype = torch.float64)
vB = torch.tensor([1.0, meanValue], dtype = torch.float64)

In [ ]:
# Objective Function

def ObjFun( vP: Tensor ) -> torch.FloatType:

    return torch.special.entr(vP).sum()

In [ ]:
# Projection Function

def ProjLinearEquality( vP: Tensor, mA: Tensor, vB: Tensor ) -> Tensor:
    """
    Projects the vector vP onto the linear equality constraint defined by mA and vB.
    
    Parameters:
    - vP: The vector to be projected.
    - mA: The matrix defining the linear equality constraint.
    - vB: The vector defining the linear equality constraint.
    
    Returns:
    - vPProj: The projected vector.
    """
    
    return vP - mA.T @ torch.linalg.solve(mA @ mA.T, (mA @ vP - vB)) #<! Can be optimized using Cholesky decomposition for better performance

In [ ]:
# Projected Gradient Descent

vP = torch.ones(numElements, dtype = torch.float64) / numElements #<! Initial distribution (Uniform)

for kk in range(numIter):
    # Compute the gradient of the objective function
    vGrad = torch.autograd.functional.jacobian(ObjFun, vP)
    
    # Update the distribution using gradient ascent
    vP = vP + μ * vGrad
    
    # Project the updated distribution onto the feasible set
    vP = ProjLinearEquality(vP, mA, vB)
    
    # Ensure non negativity
    vP = torch.clamp(vP, min = 0)

In [ ]:
# # Projected Gradient Descent - Alternative Implementation
# vP = torch.full((numElements,), 1.0 / numElements, dtype = torch.float64, requires_grad = True) #<! Initial distribution (Uniform)

# for kk in range(numIter):
#     objVal = ObjFun(vP)
#     objVal.backward() #<! Compute gradients

#     # Keep vP as a leaf tensor; otherwise vP.grad will not be populated.
#     with torch.no_grad():
#         vP += μ * vP.grad #<! Update the distribution using gradient ascent

#         # Project the updated distribution onto the feasible set
#         vP.copy_(ProjLinearEquality(vP, mA, vB))

#         # Ensure non negativity
#         vP.clamp_(min = 0)

#     vP.grad = None

In [ ]:
# Solution Summary

vP = vP.detach().numpy()  # Convert to NumPy array for summary

print(f'Optimal Distribution: {vP}')
print(f'Non Negativity: {np.all(vP >= 0)}')
print(f'Unit Sum: {np.sum(vP)}')
print(f'Mean Value: {vP @ vVals}')
print(f'Entropy: {sp.stats.entropy(vP)}')